In [1]:
import argparse
import datetime
import glob
import json
import os
import sys
import time
import traceback
import unicodedata

import numpy as np

import pandas as pd
import requests

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# ── Standardize team names in ALL games_live2 CSV files ──
# Covers filenames AND content columns (away_team_abbrev, home_team_abbrev)

HIST_TO_MODERN = {
    # Relocations / renames
    "SEA": "OKC",   # SuperSonics -> Thunder
    "VAN": "MEM",   # Vancouver Grizzlies -> Memphis
    "NJN": "BKN",   # New Jersey Nets -> Brooklyn
    "NJ":  "BKN",   # New Jersey Nets (short form)
    "BRK": "BKN",   # Brooklyn alt code
    "NOH": "NOP",   # New Orleans Hornets -> Pelicans
    "NOK": "NOP",   # New Orleans/Oklahoma City Hornets -> Pelicans
    "NO":  "NOP",   # New Orleans (short form)

    # Charlotte franchise codes across eras
    "CHH": "CHA",
    "CHO": "CHA",
    "CHA": "CHA",

    # Washington historical codes
    "WSB": "WAS",
    "WSH": "WAS",   # common ESPN-style code
    "BAL": "WAS",
    "WAS": "WAS",

    # Common abbreviation variants (ESPN / thesports style)
    "PHO": "PHX",
    "GS":  "GSW",
    "SA":  "SAS",
    "NY":  "NYK",
    "BK":  "BKN",
    "UTAH": "UTA",

    # Modern 30 teams (identity mappings for completeness)
    "ATL": "ATL", "BOS": "BOS", "BKN": "BKN", "CHI": "CHI", "CLE": "CLE",
    "DAL": "DAL", "DEN": "DEN", "DET": "DET", "GSW": "GSW", "HOU": "HOU",
    "IND": "IND", "LAC": "LAC", "LAL": "LAL", "MEM": "MEM", "MIA": "MIA",
    "MIL": "MIL", "MIN": "MIN", "NOP": "NOP", "NYK": "NYK", "OKC": "OKC",
    "ORL": "ORL", "PHI": "PHI", "PHX": "PHX", "POR": "POR", "SAC": "SAC",
    "SAS": "SAS", "TOR": "TOR", "UTA": "UTA",
}

# All-Star / special event codes to skip (leave as-is)
SPECIAL_TEAMS = {"EAST", "WEST", "USA", "WORLD", "CAN", "DUR", "GIA",
                 "KEN", "LEB", "CHK", "SHQ", "STE"}

def canonical(abbrev):
    """Map any team abbreviation to its modern canonical form."""
    if abbrev is None or (isinstance(abbrev, float) and np.isnan(abbrev)):
        return abbrev
    a = str(abbrev).strip().upper()
    if a in SPECIAL_TEAMS:
        return a  # leave All-Star teams untouched
    return HIST_TO_MODERN.get(a, a)

# Team-related content columns to standardize inside each CSV
TEAM_COLS = ["away_team_abbrev", "home_team_abbrev",
             "away_team_name_alt", "home_team_name_alt"]

# Load all schedule files to create game_id -> game_date mapping
schedule_files = glob.glob("data/schedules/schedule_*.csv")
print(f"Loading {len(schedule_files)} schedule files...")

game_date_map = {}
for sched_file in schedule_files:
    df = pd.read_csv(sched_file, dtype={"GAME_ID": str})
    for _, row in df.iterrows():
        game_id = row["GAME_ID"].zfill(10)
        game_date_map[game_id] = row["GAME_DATE"]

print(f"Loaded {len(game_date_map)} games with dates")
print(f"Sample: {list(game_date_map.items())[:3]}")

Loading 25 schedule files...
Loaded 32091 games with dates
Sample: [('0290614019', '2009-06-14'), ('0290611019', '2009-06-11'), ('0290609019', '2009-06-09')]


In [3]:
SCHEDULE = 'data/schedules/schedule_2025-26.csv'
DATE_FROM = '2026-04-05'   # Start date (inclusive)
DATE_TO   = '2026-04-06'   # End date (inclusive). Set equal to DATE_FROM for a single day.

schedule = pd.read_csv(SCHEDULE, dtype={'GAME_ID': str})
schedule['GAME_DATE'] = pd.to_datetime(schedule['GAME_DATE']).dt.strftime('%Y-%m-%d')

games_today = schedule[
    (schedule['GAME_DATE'] >= DATE_FROM) & (schedule['GAME_DATE'] <= DATE_TO)
][[
    'GAME_ID', 'GAME_DATE', 'game_date_time',
    'away_abbreviation', 'away_display_name',
    'home_abbreviation', 'home_display_name',
    'MATCHUP', 'status_type_description',
]].copy()

print(f"Date range: {DATE_FROM} to {DATE_TO}")
print(f"Games found: {len(games_today)}")

OUTPUT_CSV = f'team_stats_{DATE_FROM}_to_{DATE_TO}.csv' if DATE_FROM != DATE_TO else f'team_stats_{DATE_FROM}.csv'
SLEEP_S = 0.6
MAX_RETRIES = 5

ESPN_SUMMARY_URL = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba/summary"
ESPN_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
}

# Derive season label from schedule filename
SEASON = os.path.basename(SCHEDULE).replace('schedule_', '').replace('.csv', '')

EXPECTED_COLS = [
    'game_id', 'game_date', 'season', 'home_team', 'away_team',
    'home_team_full', 'away_team_full',
    'away_TEAM_CITY', 'away_FGM', 'away_FGA', 'away_FG_PCT',
    'away_FG3M', 'away_FG3A', 'away_FG3_PCT', 'away_FTM', 'away_FTA',
    'away_FT_PCT', 'away_OREB', 'away_DREB', 'away_REB', 'away_AST',
    'away_STL', 'away_BLK', 'away_TO', 'away_PF', 'away_PTS',
    'home_TEAM_CITY', 'home_FGM', 'home_FGA', 'home_FG_PCT',
    'home_FG3M', 'home_FG3A', 'home_FG3_PCT', 'home_FTM', 'home_FTA',
    'home_FT_PCT', 'home_OREB', 'home_DREB', 'home_REB', 'home_AST',
    'home_STL', 'home_BLK', 'home_TO', 'home_PF', 'home_PTS',
]


def parse_espn_team_stats(team_block, score):
    """Parse ESPN team statistics block into a flat dict with V2-style column names."""
    stats = {}
    for s in team_block.get('statistics', []):
        name = s['name']
        val = s['displayValue']

        # Combined stats like "fieldGoalsMade-fieldGoalsAttempted" = "50-98"
        if name == 'fieldGoalsMade-fieldGoalsAttempted':
            parts = val.split('-')
            stats['FGM'] = int(parts[0])
            stats['FGA'] = int(parts[1])
        elif name == 'fieldGoalPct':
            stats['FG_PCT'] = float(val) / 100.0
        elif name == 'threePointFieldGoalsMade-threePointFieldGoalsAttempted':
            parts = val.split('-')
            stats['FG3M'] = int(parts[0])
            stats['FG3A'] = int(parts[1])
        elif name == 'threePointFieldGoalPct':
            stats['FG3_PCT'] = float(val) / 100.0
        elif name == 'freeThrowsMade-freeThrowsAttempted':
            parts = val.split('-')
            stats['FTM'] = int(parts[0])
            stats['FTA'] = int(parts[1])
        elif name == 'freeThrowPct':
            stats['FT_PCT'] = float(val) / 100.0
        elif name == 'offensiveRebounds':
            stats['OREB'] = int(val)
        elif name == 'defensiveRebounds':
            stats['DREB'] = int(val)
        elif name == 'totalRebounds':
            stats['REB'] = int(val)
        elif name == 'assists':
            stats['AST'] = int(val)
        elif name == 'steals':
            stats['STL'] = int(val)
        elif name == 'blocks':
            stats['BLK'] = int(val)
        elif name == 'turnovers':
            stats['TO'] = int(val)
        elif name == 'fouls':
            stats['PF'] = int(val)

    # Score comes from header.competitions.competitors, not team stats
    stats['PTS'] = int(score)

    # Team city from team info
    team_info = team_block.get('team', {})
    stats['TEAM_CITY'] = team_info.get('location', team_info.get('shortDisplayName', ''))

    return stats


def fetch_box_score(event_id, sleep_s=SLEEP_S, max_retries=MAX_RETRIES):
    """Fetch team box score from ESPN Summary API, return dict with V2-style column names."""
    last_err = None
    headers = {**ESPN_HEADERS, "Referer": f"https://www.espn.com/nba/game?gameId={event_id}"}

    for attempt in range(max_retries):
        try:
            if attempt > 0:
                delay = sleep_s * (2 ** attempt)
                print(f"    [RETRY {attempt}/{max_retries}] waiting {delay:.1f}s...")
                time.sleep(delay)

            resp = requests.get(ESPN_SUMMARY_URL, params={"event": event_id},
                                headers=headers, timeout=15)
            resp.raise_for_status()
            data = resp.json()

            # Get scores from header
            comps = data.get('header', {}).get('competitions', [{}])
            scores = {}  # homeAway -> score
            for c in comps[0].get('competitors', []):
                scores[c.get('homeAway')] = c.get('score', '0')

            # Get team stats from boxscore
            teams_blocks = data.get('boxscore', {}).get('teams', [])
            if len(teams_blocks) < 2:
                return None

            result = {}
            for tb in teams_blocks:
                ha = tb.get('homeAway', '')
                score = scores.get(ha, '0')
                result[ha] = parse_espn_team_stats(tb, score)

            if 'away' not in result or 'home' not in result:
                return None

            time.sleep(sleep_s)
            return result
        except Exception as e:
            last_err = e
            if attempt == max_retries - 1:
                break

    print(f"  [WARN] Failed game {event_id} after {max_retries} attempts: {last_err}")
    return None


rows = []
for _, r in games_today.iterrows():
    game_id = r['GAME_ID']
    print(f"Fetching {game_id}: {r['away_abbreviation']} @ {r['home_abbreviation']} ...", end=' ')

    box = fetch_box_score(game_id)
    if box is None:
        print("SKIP")
        continue

    row = {
        'game_id':        game_id,
        'game_date':      r['GAME_DATE'],
        'season':         SEASON,
        'home_team':      r['home_abbreviation'],
        'away_team':      r['away_abbreviation'],
        'home_team_full': r['home_display_name'],
        'away_team_full': r['away_display_name'],
    }
    for k, v in box['away'].items():
        row[f'away_{k}'] = v
    for k, v in box['home'].items():
        row[f'home_{k}'] = v

    rows.append(row)
    print("OK")

Date range: 2026-04-05 to 2026-04-06
Games found: 16
Fetching 401811002: POR @ DEN ... OK
Fetching 401811000: CLE @ MEM ... OK
Fetching 401811001: PHI @ SA ... OK
Fetching 401810998: NY @ ATL ... OK
Fetching 401810999: DET @ ORL ... OK
Fetching 401810997: HOU @ GS ... OK
Fetching 401810996: LAC @ SAC ... OK
Fetching 401810995: LAL @ DAL ... OK
Fetching 401810992: CHA @ MIN ... OK
Fetching 401810993: ORL @ NO ... OK
Fetching 401810994: UTAH @ OKC ... OK
Fetching 401810991: IND @ CLE ... OK
Fetching 401810987: TOR @ BOS ... OK
Fetching 401810988: WSH @ BKN ... OK
Fetching 401810989: PHX @ CHI ... OK
Fetching 401810990: MEM @ MIL ... OK


In [4]:
TEAM_STATS_CSV = 'data/team_stats.csv'

if not rows:
    print("No new rows to append.")
else:
    new_df = pd.DataFrame(rows).reindex(columns=EXPECTED_COLS)

    if os.path.exists(TEAM_STATS_CSV):
        existing = pd.read_csv(TEAM_STATS_CSV, dtype={'game_id': str})
        existing_ids = set(existing['game_id'].astype(str))
        to_append = new_df[~new_df['game_id'].astype(str).isin(existing_ids)]

        if to_append.empty:
            print(f"All {len(new_df)} game(s) already present in {TEAM_STATS_CSV}. Nothing to append.")
        else:
            to_append.to_csv(TEAM_STATS_CSV, mode='a', header=False, index=False)
            print(f"Appended {len(to_append)} new game(s) to {TEAM_STATS_CSV} "
                  f"({len(new_df) - len(to_append)} already existed).")
    else:
        new_df.to_csv(TEAM_STATS_CSV, index=False)
        print(f"Created {TEAM_STATS_CSV} with {len(new_df)} game(s).")

Appended 16 new game(s) to data/team_stats.csv (0 already existed).


In [5]:
# ── Fetch game rosters from ESPN ──
PLAYER_STATS_DIR = 'data/player_stats'
GAME_ROSTERS_DIR = 'data/game_rosters'

ROSTER_COLS = [
    'game_id', 'season', 'season_type', 'game_date', 'game_date_time',
    'athlete_id', 'athlete_display_name', 'team_id', 'team_name', 'team_location',
    'team_short_display_name', 'minutes', 'field_goals_made', 'field_goals_attempted',
    'three_point_field_goals_made', 'three_point_field_goals_attempted',
    'free_throws_made', 'free_throws_attempted', 'offensive_rebounds',
    'defensive_rebounds', 'rebounds', 'assists', 'steals', 'blocks', 'turnovers', 'fouls',
    'plus_minus', 'points', 'starter', 'ejected', 'did_not_play', 'reason', 'active',
    'athlete_jersey', 'athlete_short_name', 'athlete_headshot_href',
    'athlete_position_name', 'athlete_position_abbreviation', 'team_display_name',
    'team_uid', 'team_slug', 'team_logo', 'team_abbreviation', 'home_away', 'team_winner',
    'team_score', 'opponent_team_id', 'opponent_team_name', 'opponent_team_location',
    'opponent_team_display_name', 'opponent_team_abbreviation', 'opponent_team_logo',
    'opponent_team_color', 'opponent_team_alternate_color', 'opponent_team_score',
    'team_color', 'team_alternate_color', 'schedule_season_end',
    'schedule_away_team', 'schedule_home_team',
]

def parse_split_stat(val):
    """Parse 'X-Y' stat like '10-19' into (made, attempted)."""
    if not val or val == '--':
        return (0, 0)
    parts = str(val).split('-')
    if len(parts) == 2:
        try:
            return (int(parts[0]), int(parts[1]))
        except ValueError:
            return (0, 0)
    return (0, 0)

def parse_int_stat(val):
    if not val or val == '--':
        return 0
    try:
        return int(val)
    except (ValueError, TypeError):
        return 0

def parse_plus_minus(val):
    if not val or val == '--':
        return 0
    try:
        return int(str(val).replace('+', ''))
    except (ValueError, TypeError):
        return 0

def fetch_game_roster(event_id, game_meta):
    """Fetch player box scores from ESPN Summary API. Returns list of row dicts."""
    headers = {**ESPN_HEADERS, "Referer": f"https://www.espn.com/nba/game?gameId={event_id}"}

    for attempt in range(MAX_RETRIES):
        try:
            if attempt > 0:
                time.sleep(SLEEP_S * (2 ** attempt))

            resp = requests.get(ESPN_SUMMARY_URL, params={"event": event_id},
                                headers=headers, timeout=15)
            resp.raise_for_status()
            data = resp.json()

            # Map team IDs to home/away from header
            comps = data.get('header', {}).get('competitions', [{}])[0]
            ha_by_tid = {}   # team_id_str -> 'home'/'away'
            team_meta = {}   # 'home'/'away' -> {id, score, winner, team info}

            for c in comps.get('competitors', []):
                tid = str(c.get('id', ''))
                ha = c.get('homeAway', '')
                ha_by_tid[tid] = ha
                t = c.get('team', {})
                team_meta[ha] = {
                    'team_id': tid,
                    'team_score': c.get('score', '0'),
                    'team_winner': c.get('winner', False),
                    'team_name': t.get('name', ''),
                    'team_location': t.get('location', ''),
                    'team_abbreviation': t.get('abbreviation', ''),
                    'team_display_name': t.get('displayName', ''),
                    'team_short_display_name': t.get('shortDisplayName', ''),
                    'team_uid': t.get('uid', ''),
                    'team_slug': t.get('slug', ''),
                    'team_logo': t.get('logo', ''),
                    'team_color': t.get('color', ''),
                    'team_alternate_color': t.get('alternateColor', ''),
                }

            # Parse player data from boxscore.players
            players_blocks = data.get('boxscore', {}).get('players', [])
            roster_rows = []

            for pb in players_blocks:
                team = pb.get('team', {})
                tid = str(team.get('id', ''))
                ha = ha_by_tid.get(tid, '')
                opp_ha = 'away' if ha == 'home' else 'home'
                my_meta = team_meta.get(ha, {})
                opp_meta = team_meta.get(opp_ha, {})

                for stat_block in pb.get('statistics', []):
                    names = stat_block.get('names', [])
                    name_to_idx = {n: i for i, n in enumerate(names)}

                    for ab in stat_block.get('athletes', []):
                        athlete = ab.get('athlete', {})
                        stats_arr = ab.get('stats', [])
                        dnp = ab.get('didNotPlay', False)
                        pos = athlete.get('position', {})

                        def get_s(name):
                            idx = name_to_idx.get(name)
                            if idx is not None and idx < len(stats_arr):
                                return stats_arr[idx]
                            return ''

                        row = {col: '' for col in ROSTER_COLS}

                        # Game metadata
                        row['game_id'] = event_id
                        row['season'] = game_meta.get('year', '')
                        row['season_type'] = 2
                        row['game_date'] = game_meta.get('game_date', '')
                        row['game_date_time'] = game_meta.get('game_date_time', '')

                        # Athlete info
                        row['athlete_id'] = athlete.get('id', '')
                        row['athlete_display_name'] = athlete.get('displayName', '')
                        row['athlete_jersey'] = athlete.get('jersey', '')
                        row['athlete_short_name'] = athlete.get('shortName', '')
                        row['athlete_headshot_href'] = athlete.get('headshot', {}).get('href', '')
                        row['athlete_position_name'] = pos.get('name', '')
                        row['athlete_position_abbreviation'] = pos.get('abbreviation', '')

                        # Team info
                        row['team_id'] = tid
                        row['team_name'] = my_meta.get('team_name', '')
                        row['team_location'] = my_meta.get('team_location', '')
                        row['team_short_display_name'] = my_meta.get('team_short_display_name', '')
                        row['team_display_name'] = my_meta.get('team_display_name', '')
                        row['team_uid'] = my_meta.get('team_uid', '')
                        row['team_slug'] = my_meta.get('team_slug', '')
                        row['team_logo'] = my_meta.get('team_logo', '')
                        row['team_abbreviation'] = my_meta.get('team_abbreviation', '')
                        row['team_color'] = my_meta.get('team_color', '')
                        row['team_alternate_color'] = my_meta.get('team_alternate_color', '')
                        row['home_away'] = ha
                        row['team_winner'] = my_meta.get('team_winner', '')
                        row['team_score'] = my_meta.get('team_score', '')

                        # Opponent info
                        row['opponent_team_id'] = opp_meta.get('team_id', '')
                        row['opponent_team_name'] = opp_meta.get('team_name', '')
                        row['opponent_team_location'] = opp_meta.get('team_location', '')
                        row['opponent_team_display_name'] = opp_meta.get('team_display_name', '')
                        row['opponent_team_abbreviation'] = opp_meta.get('team_abbreviation', '')
                        row['opponent_team_logo'] = opp_meta.get('team_logo', '')
                        row['opponent_team_color'] = opp_meta.get('team_color', '')
                        row['opponent_team_alternate_color'] = opp_meta.get('team_alternate_color', '')
                        row['opponent_team_score'] = opp_meta.get('team_score', '')

                        # Schedule info
                        row['schedule_away_team'] = game_meta.get('away_team', '')
                        row['schedule_home_team'] = game_meta.get('home_team', '')

                        # Status flags
                        row['starter'] = ab.get('starter', False)
                        row['ejected'] = ab.get('ejected', False)
                        row['did_not_play'] = dnp
                        row['reason'] = ab.get('reason', '')
                        row['active'] = ab.get('active', True)

                        # Stats (only for players who played)
                        if not dnp and stats_arr:
                            fg = parse_split_stat(get_s('FG'))
                            tpt = parse_split_stat(get_s('3PT'))
                            ft = parse_split_stat(get_s('FT'))
                            row['minutes'] = get_s('MIN')
                            row['field_goals_made'] = fg[0]
                            row['field_goals_attempted'] = fg[1]
                            row['three_point_field_goals_made'] = tpt[0]
                            row['three_point_field_goals_attempted'] = tpt[1]
                            row['free_throws_made'] = ft[0]
                            row['free_throws_attempted'] = ft[1]
                            row['offensive_rebounds'] = parse_int_stat(get_s('OREB'))
                            row['defensive_rebounds'] = parse_int_stat(get_s('DREB'))
                            row['rebounds'] = parse_int_stat(get_s('REB'))
                            row['assists'] = parse_int_stat(get_s('AST'))
                            row['steals'] = parse_int_stat(get_s('STL'))
                            row['blocks'] = parse_int_stat(get_s('BLK'))
                            row['turnovers'] = parse_int_stat(get_s('TO'))
                            row['fouls'] = parse_int_stat(get_s('PF'))
                            row['plus_minus'] = parse_plus_minus(get_s('+/-'))
                            row['points'] = parse_int_stat(get_s('PTS'))

                        roster_rows.append(row)

            time.sleep(SLEEP_S)
            return roster_rows
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print(f"FAIL: {e}")
                return None
    return None


# Identify games needing rosters from games_today (Cell 2).
# At this point training_games hasn't been computed yet, so we use
# the schedule-based games_today DataFrame directly.
games_needing_rosters = []
for _, row in games_today.iterrows():
    gid = str(row['GAME_ID'])
    print(f"{GAME_ROSTERS_DIR}/*/{gid}_*.csv")
    if not glob.glob(f"{GAME_ROSTERS_DIR}/*/{gid}_*.csv"):
        games_needing_rosters.append(row)

if not games_needing_rosters:
    print("All games in date range already have roster files.")
else:
    print(f"Need to fetch rosters for {len(games_needing_rosters)} game(s)")

    games_fetched = 0

    for game_row in games_needing_rosters:
        event_id = str(game_row['GAME_ID'])
        game_date = game_row['GAME_DATE']
        year = pd.to_datetime(game_date).year
        away_team = canonical(game_row['away_abbreviation'])
        home_team = canonical(game_row['home_abbreviation'])

        print(f"Fetching roster {event_id}: {away_team} @ {home_team} ...", end=' ')

        meta = {
            'game_date': game_date,
            'game_date_time': game_row.get('game_date_time', ''),
            'year': year,
            'away_team': away_team,
            'home_team': home_team,
        }

        roster_rows = fetch_game_roster(event_id, meta)
        if not roster_rows:
            print("SKIP")
            continue

        # Save roster file
        roster_df = pd.DataFrame(roster_rows).reindex(columns=ROSTER_COLS)
        year_dir = f"{GAME_ROSTERS_DIR}/{year}"
        os.makedirs(year_dir, exist_ok=True)
        roster_path = f"{year_dir}/{event_id}_{away_team}_{home_team}.csv"
        roster_df.to_csv(roster_path, index=False)

        games_fetched += 1
        print(f"OK ({len(roster_rows)} players)")

    print(f"\nFetched rosters for {games_fetched} game(s)")


data/game_rosters/*/401811002_*.csv
data/game_rosters/*/401811000_*.csv
data/game_rosters/*/401811001_*.csv
data/game_rosters/*/401810998_*.csv
data/game_rosters/*/401810999_*.csv
data/game_rosters/*/401810997_*.csv
data/game_rosters/*/401810996_*.csv
data/game_rosters/*/401810995_*.csv
data/game_rosters/*/401810992_*.csv
data/game_rosters/*/401810993_*.csv
data/game_rosters/*/401810994_*.csv
data/game_rosters/*/401810991_*.csv
data/game_rosters/*/401810987_*.csv
data/game_rosters/*/401810988_*.csv
data/game_rosters/*/401810989_*.csv
data/game_rosters/*/401810990_*.csv
Need to fetch rosters for 16 game(s)
Fetching roster 401811002: POR @ DEN ... OK (30 players)
Fetching roster 401811000: CLE @ MEM ... OK (24 players)
Fetching roster 401811001: PHI @ SAS ... OK (30 players)
Fetching roster 401810998: NYK @ ATL ... OK (30 players)
Fetching roster 401810999: DET @ ORL ... OK (26 players)
Fetching roster 401810997: HOU @ GSW ... OK (26 players)
Fetching roster 401810996: LAC @ SAC ... OK (

In [6]:
# ── Fetch missing ESPN play-by-play files → data/games_live/{season_year}/ ──
# Uses the ESPN summary API to pull PBP for games in our schedule that don't
# already have a CSV in games_live/. Output matches the 63-column format
# produced by hoopR::load_nba_pbp().

import math
from pathlib import Path


ESPN_SUMMARY_URL = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary"
PBP_OUT_DIR = Path("data/games_live")
PBP_DELAY = 0.6  # seconds between API calls

# ── Which season year folder to write into ──
SEASON_YEAR = 2026  # end-year of the season (2025-26 → 2026)
SEASON_TYPE_DEFAULT = 2  # 2 = regular season


def _espn_get_summary(event_id: str) -> dict:
    """Fetch the ESPN game summary JSON for a given event ID."""
    r = requests.get(
        ESPN_SUMMARY_URL,
        params={"event": str(event_id)},
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20,
    )
    r.raise_for_status()
    return r.json()


def _parse_clock(clock_str: str):
    """Parse 'MM:SS' into (minutes, seconds)."""
    if not clock_str or clock_str == "0:00":
        return 0, 0
    parts = clock_str.split(":")
    if len(parts) == 2:
        return int(parts[0]), int(parts[1])
    return 0, 0


def _compute_time_cols(period: int, clock_min: int, clock_sec: int):
    """Compute the 6 seconds-remaining columns + half/qtr helpers."""
    REG_QUARTER_SEC = 720
    OT_PERIOD_SEC = 300

    secs_left_in_period = clock_min * 60 + clock_sec

    if period <= 4:
        # Regulation
        qtr = period
        half = 1 if period <= 2 else 2
        quarters_left_after = 4 - period
        start_qtr_sec = secs_left_in_period
        if half == 1:
            start_half_sec = secs_left_in_period + (2 - period) * REG_QUARTER_SEC
        else:
            start_half_sec = secs_left_in_period + (4 - period) * REG_QUARTER_SEC
        start_game_sec = secs_left_in_period + quarters_left_after * REG_QUARTER_SEC
    else:
        # Overtime
        qtr = period
        half = 2
        start_qtr_sec = secs_left_in_period
        start_half_sec = secs_left_in_period
        start_game_sec = secs_left_in_period

    return {
        "qtr": qtr,
        "half": half,
        "game_half": half,
        "clock_minutes": clock_min,
        "clock_seconds": clock_sec,
        "start_quarter_seconds_remaining": start_qtr_sec,
        "start_half_seconds_remaining": start_half_sec,
        "start_game_seconds_remaining": start_game_sec,
    }


def _transform_coordinate(raw_x, raw_y):
    """Transform ESPN raw coordinates to court coordinates matching hoopR output."""
    if raw_x is None or raw_y is None:
        return None, None
    if abs(raw_x) > 10000 or abs(raw_y) > 10000:
        return None, None
    coord_x = raw_x * 2.375 - 91.5
    coord_y = raw_y - 11.0
    return round(coord_x, 2), round(coord_y, 2)


def espn_pbp_to_df(data: dict, event_id: str) -> pd.DataFrame:
    """Convert ESPN summary JSON to a DataFrame matching the 63-column games_live format."""
    plays_raw = data.get("plays", [])
    if not plays_raw:
        return pd.DataFrame()

    # Extract team metadata from header
    comp = data["header"]["competitions"][0]
    competitors = {c["homeAway"]: c for c in comp["competitors"]}

    home_c = competitors["home"]
    away_c = competitors["away"]

    home_team_id = int(home_c["id"])
    home_team_abbrev = home_c["team"]["abbreviation"]
    home_team_name = home_c["team"]["location"] if "location" in home_c["team"] else home_c["team"].get("displayName", "")
    home_team_mascot = home_c["team"].get("name", "")
    home_team_name_alt = home_c["team"].get("shortDisplayName", home_team_name)

    away_team_id = int(away_c["id"])
    away_team_abbrev = away_c["team"]["abbreviation"]
    away_team_name = away_c["team"]["location"] if "location" in away_c["team"] else away_c["team"].get("displayName", "")
    away_team_mascot = away_c["team"].get("name", "")
    away_team_name_alt = away_c["team"].get("shortDisplayName", away_team_name)

    # Game date/time
    game_dt_str = comp.get("date", "")
    game_date = game_dt_str[:10] if game_dt_str else ""

    # Season info
    season = SEASON_YEAR
    season_type = SEASON_TYPE_DEFAULT

    # Spread info from pickcenter
    game_spread = None
    home_favorite = None
    game_spread_available = False
    home_team_spread = None
    if "pickcenter" in data and data["pickcenter"]:
        pc = data["pickcenter"][0]
        spread_val = pc.get("spread")
        if spread_val is not None:
            game_spread = abs(float(spread_val))
            game_spread_available = True
            home_odds = pc.get("homeTeamOdds", {})
            home_favorite = home_odds.get("favorite", False)
            home_team_spread = float(spread_val) if home_favorite else -float(spread_val)

    rows = []

    for play_num, play in enumerate(plays_raw, 1):
        period_info = play.get("period", {})
        period_number = period_info.get("number", 1)
        period_display = period_info.get("displayValue", f"Quarter {period_number}")

        clock_str = play.get("clock", {}).get("displayValue", "0:00")
        clock_min, clock_sec = _parse_clock(clock_str)

        time_cols = _compute_time_cols(period_number, clock_min, clock_sec)

        # Coordinates
        coord = play.get("coordinate", {})
        raw_x = coord.get("x")
        raw_y = coord.get("y")
        cx, cy = _transform_coordinate(raw_x, raw_y)

        # Participants (up to 3 athlete IDs)
        participants = play.get("participants", [])
        ath1 = int(participants[0]["athlete"]["id"]) if len(participants) > 0 else None
        ath2 = int(participants[1]["athlete"]["id"]) if len(participants) > 1 else None
        ath3 = int(participants[2]["athlete"]["id"]) if len(participants) > 2 else None

        # Team ID for this play
        team_id = play.get("team", {}).get("id")
        if team_id is not None:
            team_id = int(team_id)

        play_type = play.get("type", {})

        lead_qtr = play_num
        lead_half = play_num

        end_qtr = time_cols["start_quarter_seconds_remaining"]
        end_half = time_cols["start_half_seconds_remaining"]
        end_game = time_cols["start_game_seconds_remaining"]

        row = {
            "game_play_number": play_num,
            "id": play.get("id"),
            "sequence_number": play.get("sequenceNumber"),
            "type_id": play_type.get("id"),
            "type_text": play_type.get("text"),
            "text": play.get("text"),
            "away_score": play.get("awayScore", 0),
            "home_score": play.get("homeScore", 0),
            "period_number": period_number,
            "period_display_value": period_display,
            "clock_display_value": clock_str,
            "scoring_play": play.get("scoringPlay", False),
            "score_value": play.get("scoreValue", 0),
            "team_id": team_id,
            "athlete_id_1": ath1,
            "athlete_id_2": ath2,
            "athlete_id_3": ath3,
            "wallclock": play.get("wallclock"),
            "shooting_play": play.get("shootingPlay", False),
            "coordinate_x_raw": raw_x if raw_x is not None and abs(raw_x) < 10000 else None,
            "coordinate_y_raw": raw_y if raw_y is not None and abs(raw_y) < 10000 else None,
            "points_attempted": play.get("pointsAttempted", 0),
            "short_description": play.get("shortDescription"),
            "season": season,
            "season_type": season_type,
            "home_team_id": home_team_id,
            "home_team_name": home_team_name,
            "home_team_mascot": home_team_mascot,
            "home_team_abbrev": home_team_abbrev,
            "home_team_name_alt": home_team_name_alt,
            "away_team_id": away_team_id,
            "away_team_name": away_team_name,
            "away_team_mascot": away_team_mascot,
            "away_team_abbrev": away_team_abbrev,
            "away_team_name_alt": away_team_name_alt,
            "game_spread": game_spread,
            "home_favorite": home_favorite,
            "game_spread_available": game_spread_available,
            "home_team_spread": home_team_spread,
            "qtr": time_cols["qtr"],
            "time": clock_str,
            "clock_minutes": time_cols["clock_minutes"],
            "clock_seconds": time_cols["clock_seconds"],
            "home_timeout_called": False,
            "away_timeout_called": False,
            "half": time_cols["half"],
            "game_half": time_cols["game_half"],
            "lead_qtr": lead_qtr,
            "lead_half": lead_half,
            "start_quarter_seconds_remaining": time_cols["start_quarter_seconds_remaining"],
            "start_half_seconds_remaining": time_cols["start_half_seconds_remaining"],
            "start_game_seconds_remaining": time_cols["start_game_seconds_remaining"],
            "end_quarter_seconds_remaining": end_qtr,
            "end_half_seconds_remaining": end_half,
            "end_game_seconds_remaining": end_game,
            "period": period_number,
            "lag_qtr": None,
            "lag_half": None,
            "coordinate_x": cx,
            "coordinate_y": cy,
            "game_date": game_date,
            "game_date_time": game_dt_str,
            "type_abbreviation": play_type.get("abbreviation"),
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    # Compute end_* from next play's start_* (shift by -1)
    for prefix in ["quarter", "half", "game"]:
        start_col = f"start_{prefix}_seconds_remaining"
        end_col = f"end_{prefix}_seconds_remaining"
        df[end_col] = df[start_col].shift(-1)
        df.loc[df.index[-1], end_col] = df.loc[df.index[-1], start_col]

    # Compute lag columns (previous play's qtr/half)
    df["lag_qtr"] = df["qtr"].shift(1)
    df["lag_half"] = df["half"].shift(1)

    # Detect timeout plays and set flags
    timeout_mask = df["type_text"].str.contains("Timeout", case=False, na=False)
    for idx in df.index[timeout_mask]:
        tid = df.loc[idx, "team_id"]
        if tid == home_team_id:
            df.loc[idx, "home_timeout_called"] = True
        elif tid == away_team_id:
            df.loc[idx, "away_timeout_called"] = True

    return df


# ── Main: find missing games and fetch ──

sched = pd.read_csv(SCHEDULE, dtype={"GAME_ID": str})
sched["GAME_DATE"] = pd.to_datetime(sched["date"]).dt.normalize()

# Filter to date range; treat games in the past as completed even if
# the schedule CSV hasn't been refreshed (status_type_completed may be stale)
today = pd.Timestamp.now().normalize()
sched_filtered = sched[
    (sched["GAME_DATE"] >= pd.to_datetime(DATE_FROM))
    & (sched["GAME_DATE"] <= pd.to_datetime(DATE_TO))
    & (sched["GAME_DATE"] < today)  # only past games
].copy()

# Exclude All-Star / non-standard games
ALL_STAR_IDS = {"STARS", "STRIPES", "WORLD", "EAST", "WEST", "USA",
                "LEB", "GIA", "DUR", "STE", "CHK", "SHQ", "CAN", "KEN"}
if "home_team_abbreviation" in sched_filtered.columns:
    mask = (
        sched_filtered["home_team_abbreviation"].isin(ALL_STAR_IDS)
        | sched_filtered["away_team_abbreviation"].isin(ALL_STAR_IDS)
    )
    sched_filtered = sched_filtered[~mask]

print(f"Schedule games in [{DATE_FROM}, {DATE_TO}] (past only): {len(sched_filtered)}")

# Find which games already have PBP files
season_dir = PBP_OUT_DIR / str(SEASON_YEAR)
season_dir.mkdir(parents=True, exist_ok=True)

existing_pbp_ids = set()
for f in season_dir.glob("*.csv"):
    existing_pbp_ids.add(f.stem.split("_")[0])

missing = sched_filtered[~sched_filtered["GAME_ID"].isin(existing_pbp_ids)].copy()
print(f"Already have PBP: {len(sched_filtered) - len(missing)}")
print(f"Missing PBP to fetch: {len(missing)}")

if len(missing) == 0:
    print("Nothing to fetch!")
else:
    success = 0
    errors = 0

    for i, (_, row) in enumerate(missing.iterrows(), 1):
        event_id = row["GAME_ID"]
        game_date = row["GAME_DATE"]

        print(f"  [{i}/{len(missing)}] {game_date.date()} (ESPN {event_id}) ...", end=" ")

        try:
            data = _espn_get_summary(event_id)

            # Get team abbrevs from API response for filename
            comp = data["header"]["competitions"][0]
            competitors = {c["homeAway"]: c for c in comp["competitors"]}
            away_abbr = competitors["away"]["team"]["abbreviation"]
            home_abbr = competitors["home"]["team"]["abbreviation"]

            df = espn_pbp_to_df(data, event_id)

            if df.empty:
                print(f"SKIP (no plays) {away_abbr} @ {home_abbr}")
                continue

            out_path = season_dir / f"{event_id}_{away_abbr}_{home_abbr}.csv"
            df.to_csv(out_path, index=False)
            print(f"OK {away_abbr} @ {home_abbr} ({len(df)} plays)")
            success += 1

        except Exception as e:
            print(f"ERROR: {e}")
            errors += 1

        time.sleep(PBP_DELAY)

    print(f"\nDone. Fetched: {success}, Errors: {errors}")

Schedule games in [2026-04-05, 2026-04-06] (past only): 13
Already have PBP: 0
Missing PBP to fetch: 13
  [1/13] 2026-04-06 (ESPN 401810998) ... OK NY @ ATL (475 plays)
  [2/13] 2026-04-06 (ESPN 401810999) ... OK DET @ ORL (495 plays)
  [3/13] 2026-04-06 (ESPN 401810997) ... OK HOU @ GS (456 plays)
  [4/13] 2026-04-06 (ESPN 401810996) ... OK LAC @ SAC (448 plays)
  [5/13] 2026-04-05 (ESPN 401810995) ... OK LAL @ DAL (503 plays)
  [6/13] 2026-04-05 (ESPN 401810992) ... OK CHA @ MIN (474 plays)
  [7/13] 2026-04-05 (ESPN 401810993) ... OK ORL @ NO (533 plays)
  [8/13] 2026-04-05 (ESPN 401810994) ... OK UTAH @ OKC (450 plays)
  [9/13] 2026-04-05 (ESPN 401810991) ... OK IND @ CLE (480 plays)
  [10/13] 2026-04-05 (ESPN 401810987) ... OK TOR @ BOS (454 plays)
  [11/13] 2026-04-05 (ESPN 401810988) ... OK WSH @ BKN (443 plays)
  [12/13] 2026-04-05 (ESPN 401810989) ... OK PHX @ CHI (497 plays)
  [13/13] 2026-04-05 (ESPN 401810990) ... OK MEM @ MIL (471 plays)

Done. Fetched: 13, Errors: 0


In [7]:
TRAINING_CSV = 'data/training_games.csv'

# Load existing training_games and team_stats
training = pd.read_csv(TRAINING_CSV, dtype={'game_id': str})
team_stats_all = pd.read_csv(TEAM_STATS_CSV, dtype={'game_id': str})

# Find new games: in team_stats but not yet in training_games
existing_ids = set(training['game_id'].astype(str))
new_games = team_stats_all[~team_stats_all['game_id'].astype(str).isin(existing_ids)].copy()

# Drop games where either team isn't in HIST_TO_MODERN
known_teams = set(HIST_TO_MODERN.keys())
unmapped_mask = ~(
    new_games['home_team'].str.upper().isin(known_teams) &
    new_games['away_team'].str.upper().isin(known_teams)
)
if unmapped_mask.any():
    skipped = new_games[unmapped_mask][['game_id', 'home_team', 'away_team']]
    print(f"Skipping {len(skipped)} game(s) with unmapped team abbreviation(s):")
    print(skipped.to_string(index=False))
    new_games = new_games[~unmapped_mask].copy()

if new_games.empty:
    print("No new games to add to training_games.csv.")
else:
    print(f"Found {len(new_games)} new game(s) to process")

    # Rebuild current season records from existing training_games data.
    # Only replay seasons that appear in the new games.
    new_seasons = set(new_games['season'].unique())
    season_records = {}  # {season: {team: {'wins': int, 'losses': int}}}

    for season in new_seasons:
        season_records[season] = {}
        season_games = training[training['season'] == season]

        for _, row in season_games.iterrows():
            home = row['home_team']
            away = row['away_team']

            for t in [home, away]:
                if t not in season_records[season]:
                    season_records[season][t] = {'wins': 0, 'losses': 0}

            home_pts = row['home_PTS']
            away_pts = row['away_PTS']

            if pd.notna(home_pts) and pd.notna(away_pts):
                if float(home_pts) > float(away_pts):
                    season_records[season][home]['wins'] += 1
                    season_records[season][away]['losses'] += 1
                elif float(away_pts) > float(home_pts):
                    season_records[season][away]['wins'] += 1
                    season_records[season][home]['losses'] += 1

    print(f"Rebuilt records for season(s): {sorted(new_seasons)}")
    for season in sorted(new_seasons):
        sample_teams = list(season_records[season].items())[:3]
        print(f"  {season} sample: {sample_teams}")

    # Sort new games chronologically (important: records must be assigned in order)
    new_games['game_date'] = pd.to_datetime(new_games['game_date'])
    new_games = new_games.sort_values('game_date').reset_index(drop=True)

    # Assign pre-game records and update after each game
    new_games['home_wins'] = 0
    new_games['home_losses'] = 0
    new_games['away_wins'] = 0
    new_games['away_losses'] = 0

    # -- Build last-game-date lookup for days_rest --
    # Combine existing training data + all games to track each team's last game
    MAX_REST = 7
    training['game_date'] = pd.to_datetime(training['game_date'])
    all_dates = training[['game_date', 'home_team', 'away_team']].copy()

    # For each team, find the most recent game date from existing data
    team_last_game = {}
    for _, row in all_dates.iterrows():
        gd = row['game_date']
        for t in [row['home_team'], row['away_team']]:
            if t not in team_last_game or gd > team_last_game[t]:
                team_last_game[t] = gd

    new_games['home_days_rest'] = MAX_REST
    new_games['away_days_rest'] = MAX_REST

    for idx, row in new_games.iterrows():
        season = row['season']
        home = row['home_team']
        away = row['away_team']
        gd = row['game_date']

        if season not in season_records:
            season_records[season] = {}
        for t in [home, away]:
            if t not in season_records[season]:
                season_records[season][t] = {'wins': 0, 'losses': 0}

        # Assign pre-game records (no leakage)
        new_games.at[idx, 'home_wins'] = season_records[season][home]['wins']
        new_games.at[idx, 'home_losses'] = season_records[season][home]['losses']
        new_games.at[idx, 'away_wins'] = season_records[season][away]['wins']
        new_games.at[idx, 'away_losses'] = season_records[season][away]['losses']

        # Compute days rest (capped at MAX_REST)
        if home in team_last_game:
            home_rest = (gd - team_last_game[home]).days
            new_games.at[idx, 'home_days_rest'] = min(max(home_rest, 0), MAX_REST)
        if away in team_last_game:
            away_rest = (gd - team_last_game[away]).days
            new_games.at[idx, 'away_days_rest'] = min(max(away_rest, 0), MAX_REST)

        # Update last game date for both teams
        team_last_game[home] = gd
        team_last_game[away] = gd

        # Update records based on game result
        home_pts = row['home_PTS']
        away_pts = row['away_PTS']

        if pd.notna(home_pts) and pd.notna(away_pts):
            if float(home_pts) > float(away_pts):
                season_records[season][home]['wins'] += 1
                season_records[season][away]['losses'] += 1
            elif float(away_pts) > float(home_pts):
                season_records[season][away]['wins'] += 1
                season_records[season][home]['losses'] += 1

    # Ensure column order matches existing training_games.csv
    new_games = new_games.reindex(columns=training.columns)

    # Append to training_games.csv
    new_games.to_csv(TRAINING_CSV, mode='a', header=False, index=False)
    print(f"\nAppended {len(new_games)} game(s) to {TRAINING_CSV}")

    # Show results
    print(f"\nNew games added:")
    print(new_games[['game_id', 'game_date', 'home_team', 'away_team',
                      'home_PTS', 'away_PTS', 'home_wins', 'home_losses',
                      'away_wins', 'away_losses',
                      'home_days_rest', 'away_days_rest']].to_string(index=False))

Skipping 22 game(s) with unmapped team abbreviation(s):
  game_id home_team away_team
400436572      WEST      EAST
400517994      WEST      EAST
400606285      EAST      WEST
400829482      EAST      WEST
400935635      WEST      EAST
401001760       STE       LEB
401018033       USA     WORLD
401098037       GIA       LEB
401098038       USA     WORLD
401197813       GIA       LEB
401197814       USA     WORLD
401306581       DUR       LEB
401410564       LEB       DUR
401524696       LEB       GIA
401623259      EAST      WEST
401752957       CHK       SHQ
401752956       SHQ       CAN
401752955       CHK       KEN
401838140     STARS     WORLD
401838141   STRIPES     STARS
401838142   STRIPES     WORLD
401838143   STRIPES     STARS
Found 16 new game(s) to process
Rebuilt records for season(s): ['2025-26']
  2025-26 sample: [('LAL', {'wins': 50, 'losses': 27}), ('GS', {'wins': 36, 'losses': 41}), ('OKC', {'wins': 61, 'losses': 16})]

Appended 16 game(s) to data/training_games.csv

N

In [8]:
TRAINING2_CSV = 'data/training_games2.csv'

# Load source data
training_all = pd.read_csv(TRAINING_CSV, dtype={'game_id': str})
training2_existing = pd.read_csv(TRAINING2_CSV, dtype={'game_id': str})

# Find new games: in training_games but not yet in training_games2
existing_ids_t2 = set(training2_existing['game_id'].astype(str))
new_game_ids = set(training_all['game_id'].astype(str)) - existing_ids_t2

if not new_game_ids:
    print("No new games to add to training_games2.csv.")
else:
    print(f"Found {len(new_game_ids)} game(s) in training_games not in training_games2")

    # Recency-weighting config (must match data_preprocess.ipynb)
    stat_names = ['FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT',
                  'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS']
    DECAY = 0.9
    MIN_GAMES = 4
    TEAM_ROLLING_WINDOW = 10

    def compute_weighted_stats(history, stat_names, decay):
        """Recency-weighted averages: most recent game gets decay^1, next decay^2, etc."""
        n = len(history)
        games_reversed = list(reversed(history))
        raw_weights = [decay ** (t + 1) for t in range(n)]
        weight_total = sum(raw_weights)
        norm_weights = [w / weight_total for w in raw_weights]
        result = {}
        for stat in stat_names:
            weighted_sum = 0.0
            weight_used = 0.0
            for i, game in enumerate(games_reversed):
                val = game.get(stat)
                if val is not None and not np.isnan(val):
                    weighted_sum += norm_weights[i] * val
                    weight_used += norm_weights[i]
            if weight_used > 0:
                result[stat] = weighted_sum / weight_used
            else:
                result[stat] = np.nan
        return result

    def compute_rolling_stats(history, stat_names, window):
        """Simple rolling average of the last `window` games."""
        recent = history[-window:]
        result = {}
        for stat in stat_names:
            vals = [g.get(stat) for g in recent if g.get(stat) is not None and not np.isnan(g.get(stat))]
            result[stat] = np.mean(vals) if vals else np.nan
        return result

    # Identify seasons that contain new games
    new_games_df = training_all[training_all['game_id'].astype(str).isin(new_game_ids)]
    new_seasons = set(new_games_df['season'].unique())
    print(f"Seasons with new games: {sorted(new_seasons)}")

    # Replay ALL games from those seasons chronologically to build team history.
    # Only output rows for new games that pass the MIN_GAMES filter.
    season_data = training_all[training_all['season'].isin(new_seasons)].copy()
    season_data['game_date'] = pd.to_datetime(season_data['game_date'])
    season_data = season_data.sort_values('game_date').reset_index(drop=True)

    team_season_history = {}  # (team, season) -> [stat_dicts]
    new_rows = []
    skipped_insufficient = 0

    for idx, row in season_data.iterrows():
        game_id = str(row['game_id'])
        away_team = row['away_team']
        home_team = row['home_team']
        season = row['season']
        is_new = game_id in new_game_ids

        away_key = (away_team, season)
        home_key = (home_team, season)

        away_history = team_season_history.get(away_key, [])
        home_history = team_season_history.get(home_key, [])

        if is_new:
            if len(away_history) < MIN_GAMES or len(home_history) < MIN_GAMES:
                skipped_insufficient += 1
            else:
                new_row = row.to_dict()

                # Preserve original PTS before overwriting
                new_row['away_PTS_actual'] = new_row.get('away_PTS')
                new_row['home_PTS_actual'] = new_row.get('home_PTS')

                # Overwrite stat columns with recency-weighted team averages
                away_weighted = compute_weighted_stats(away_history, stat_names, DECAY)
                for stat in stat_names:
                    new_row[f'away_{stat}'] = away_weighted[stat]

                home_weighted = compute_weighted_stats(home_history, stat_names, DECAY)
                for stat in stat_names:
                    new_row[f'home_{stat}'] = home_weighted[stat]

                # 10-game rolling averages for recent form
                if len(away_history) >= TEAM_ROLLING_WINDOW:
                    away_rolling = compute_rolling_stats(away_history, stat_names, TEAM_ROLLING_WINDOW)
                    for stat in stat_names:
                        new_row[f'away_r10_{stat}'] = away_rolling[stat]
                else:
                    for stat in stat_names:
                        new_row[f'away_r10_{stat}'] = np.nan

                if len(home_history) >= TEAM_ROLLING_WINDOW:
                    home_rolling = compute_rolling_stats(home_history, stat_names, TEAM_ROLLING_WINDOW)
                    for stat in stat_names:
                        new_row[f'home_r10_{stat}'] = home_rolling[stat]
                else:
                    for stat in stat_names:
                        new_row[f'home_r10_{stat}'] = np.nan

                # Compute eFG from weighted averages (not raw box scores)
                weighted_away_FGA = float(new_row.get('away_FGA', 0) or 0)
                weighted_home_FGA = float(new_row.get('home_FGA', 0) or 0)
                new_row['away_eFG'] = ((new_row['away_FGM'] + 0.5 * new_row['away_FG3M']) / weighted_away_FGA * 100) if weighted_away_FGA > 0 else np.nan
                new_row['home_eFG'] = ((new_row['home_FGM'] + 0.5 * new_row['home_FG3M']) / weighted_home_FGA * 100) if weighted_home_FGA > 0 else np.nan

                new_rows.append(new_row)

        # Always update team history with this game's RAW stats (for future games)
        for prefix, key in [('away', away_key), ('home', home_key)]:
            if key not in team_season_history:
                team_season_history[key] = []
            game_stats = {}
            for stat in stat_names:
                col = f'{prefix}_{stat}'
                if col in row.index:
                    val = row[col]
                    try:
                        game_stats[stat] = float(val) if not pd.isna(val) else np.nan
                    except (ValueError, TypeError):
                        game_stats[stat] = np.nan
                else:
                    game_stats[stat] = np.nan
            team_season_history[key].append(game_stats)

    if not new_rows:
        print(f"No new games passed MIN_GAMES={MIN_GAMES} filter "
              f"({skipped_insufficient} skipped for insufficient history).")
    else:
        new_df = pd.DataFrame(new_rows)
        # Ensure column order matches existing training_games2.csv
        new_df = new_df.reindex(columns=training2_existing.columns)
        new_df.to_csv(TRAINING2_CSV, mode='a', header=False, index=False)
        print(f"\nAppended {len(new_df)} game(s) to {TRAINING2_CSV}")
        print(f"  Skipped {skipped_insufficient} game(s) for insufficient season history")
        print(f"\nSample new rows:")
        print(new_df[['game_id', 'game_date', 'home_team', 'away_team',
                       'away_PTS', 'home_PTS', 'away_PTS_actual', 'home_PTS_actual'
                       ]].head(5).to_string(index=False))

Found 1611 game(s) in training_games not in training_games2
Seasons with new games: ['2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2008-09', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26']

Appended 16 game(s) to data/training_games2.csv
  Skipped 1595 game(s) for insufficient season history

Sample new rows:
  game_id  game_date home_team away_team   away_PTS   home_PTS  away_PTS_actual  home_PTS_actual
401810989 2026-04-05       CHI       PHX 114.160166 116.290733              120              110
401810988 2026-04-05       BKN       WSH 115.958503 102.851901              115              121
401810987 2026-04-05       BOS       TOR 118.115354 115.955622              101              115
401810991 2026-04-05       CLE       IND 119.036102 120.017642              108              117
401810994 2026-04-05       OKC   

In [9]:
# ── Update player_stats files from fetched game rosters ──
# Reads the roster CSVs saved by the fetch cell and appends each player's
# game stats to their individual player_stats file.

players_updated = set()
games_processed = 0

for _, game_row in games_today.iterrows():
    gid = str(game_row['GAME_ID'])
    roster_files = glob.glob(f"{GAME_ROSTERS_DIR}/*/{gid}_*.csv")
    if not roster_files:
        continue

    roster_df = pd.read_csv(roster_files[0], dtype={'athlete_id': str, 'team_id': str})
    if roster_df.empty:
        continue

    game_date = game_row['GAME_DATE']
    padded_id = gid.zfill(10)
    roster_df['GAME_ID'] = padded_id
    roster_df['GAME_DATE'] = game_date

    for _, pr in roster_df.iterrows():
        pid = str(pr.get('athlete_id', ''))
        if not pid or pid == 'nan' or pid == '':
            continue

        pfile = f"{PLAYER_STATS_DIR}/{pid}.csv"
        row_df = pd.DataFrame([pr])

        if os.path.exists(pfile):
            # Check if this game is already in the player's file
            existing = pd.read_csv(pfile, dtype={'GAME_ID': str}, usecols=['GAME_ID'])
            if padded_id in existing['GAME_ID'].astype(str).values:
                continue  # already have this game
            existing_cols = pd.read_csv(pfile, nrows=0).columns.tolist()
            row_df = row_df.reindex(columns=existing_cols)
            row_df.to_csv(pfile, mode='a', header=False, index=False)
        else:
            row_df.to_csv(pfile, mode='w', header=True, index=False)

        players_updated.add(pid)

    games_processed += 1

print(f"Updated player_stats for {games_processed} game(s), {len(players_updated)} unique players")

Updated player_stats for 16 game(s), 400 unique players


In [10]:
# ── Compute 5g/10g/20g rolling player averages for new games → training_games3.csv ──
# Must match data_preprocess.ipynb: per-minute rates for count stats, full roster (no
# current-game minutes filter), prior avg minutes weighting, MIN_MINUTES_FOR_RATE = 5.

TRAINING3_CSV = 'data/training_games3.csv'

training2_all = pd.read_csv(TRAINING2_CSV, dtype={'game_id': str})
training3_existing = pd.read_csv(TRAINING3_CSV, dtype={'game_id': str})

# Use game IDs from new_games (Cell 4) instead of diffing CSVs
new_ids_t3 = set(new_games['game_id'].astype(str))
print(f"New games from today's batch: {len(new_ids_t3)}")

if not new_ids_t3:
    print("No new games to add to training_games3.csv.")
else:

    # Load team ID → abbreviation mapping (ESPN team IDs)
    teams_df = pd.read_csv("data/nba_teams.csv", dtype={"team_id": str})
    team_id_to_abbr = {tid: canonical(abbr) for tid, abbr in zip(teams_df["team_id"], teams_df["team_code"])}

    # ESPN column name → short stat name mapping
    ESPN_COL = {
        'FGA': 'field_goals_attempted', 'FG3M': 'three_point_field_goals_made',
        'FG3A': 'three_point_field_goals_attempted',
        'FTM': 'free_throws_made', 'FTA': 'free_throws_attempted',
        'OREB': 'offensive_rebounds', 'DREB': 'defensive_rebounds',
        'REB': 'rebounds', 'AST': 'assists', 'STL': 'steals',
        'BLK': 'blocks', 'TO': 'turnovers', 'PF': 'fouls',
        'PTS': 'points', 'PLUS_MINUS': 'plus_minus',
    }
    PLAYER_COUNT_COLS = ['FGA', 'FG3M', 'FG3A', 'FTM', 'FTA',
                         'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS']
    PLAYER_PCT_COLS = ['FG_PCT', 'FG3_PCT', 'FT_PCT']
    PLAYER_STAT_COLS = PLAYER_COUNT_COLS + PLAYER_PCT_COLS
    WINDOWS = [5, 10, 20]
    MIN_MINUTES_FOR_RATE = 5.0

    # Build game_id → roster file mapping for new games
    game_id_to_roster = {}
    for gid in new_ids_t3:
        rf = glob.glob(f"{GAME_ROSTERS_DIR}/*/{gid}_*.csv")
        if rf:
            game_id_to_roster[gid] = rf[0]

    # Cache for player stats files
    _player_cache = {}

    def _convert_min(val):
        if pd.isna(val) or val == '':
            return 0.0
        try:
            return float(val)
        except (ValueError, TypeError):
            pass
        s = str(val).strip()
        if ':' in s:
            parts = s.split(':')
            try:
                return float(parts[0]) + (float(parts[1]) / 60.0)
            except ValueError:
                return 0.0
        return 0.0

    def _load_player(pid):
        if pid in _player_cache:
            return _player_cache[pid]
        pfile = f"{PLAYER_STATS_DIR}/{pid}.csv"
        if not os.path.exists(pfile):
            _player_cache[pid] = None
            return None
        try:
            df = pd.read_csv(pfile, dtype={'GAME_ID': str, 'game_id': str, 'athlete_id': str, 'team_id': str})
            _player_cache[pid] = df
            return df
        except Exception:
            _player_cache[pid] = None
            return None

    def get_prior_avg_minutes(pid, cur_game_date, n=5):
        df = _load_player(pid)
        if df is None or df.empty:
            return 0.0
        date_col = 'GAME_DATE' if 'GAME_DATE' in df.columns else ('game_date' if 'game_date' in df.columns else None)
        min_col = 'minutes' if 'minutes' in df.columns else ('MIN' if 'MIN' in df.columns else None)
        if not date_col or not min_col:
            return 0.0
        d = df.copy()
        d[date_col] = pd.to_datetime(d[date_col], errors='coerce')
        d['_min'] = d[min_col].apply(_convert_min)
        cur_dt = pd.to_datetime(cur_game_date, errors='coerce')
        prior = d[(d[date_col] < cur_dt) & (d['_min'] > 0)]
        if prior.empty:
            return 0.0
        return float(prior.sort_values(date_col, ascending=False).head(n)['_min'].mean())

    def get_player_rolling(pid, cur_game_id, cur_game_date, max_window=20):
        """Per-minute rolling stats. Count stats divided by minutes; pct stats as-is."""
        df = _load_player(pid)
        if df is None or df.empty:
            return None
        date_col = 'GAME_DATE' if 'GAME_DATE' in df.columns else ('game_date' if 'game_date' in df.columns else None)
        gid_col = 'GAME_ID' if 'GAME_ID' in df.columns else ('game_id' if 'game_id' in df.columns else None)
        min_col = 'minutes' if 'minutes' in df.columns else ('MIN' if 'MIN' in df.columns else None)
        if not date_col or not gid_col or not min_col:
            return None

        d = df.copy()
        d['_min'] = d[min_col].apply(_convert_min)
        d = d[d['_min'] >= MIN_MINUTES_FOR_RATE]
        d = d[d[gid_col].astype(str) != str(cur_game_id).zfill(10)]
        if d.empty:
            return None
        d[date_col] = pd.to_datetime(d[date_col], errors='coerce')
        d = d.dropna(subset=[date_col])
        cur_dt = pd.to_datetime(cur_game_date, errors='coerce')
        if cur_dt is not None and not pd.isna(cur_dt):
            d = d[d[date_col] < cur_dt]
        d = d.sort_values(date_col, ascending=False).head(max_window)
        if d.empty:
            return None

        # Map ESPN columns to short stat names
        for short_name, espn_col in ESPN_COL.items():
            if espn_col in d.columns:
                d[short_name] = pd.to_numeric(d[espn_col], errors='coerce')
        # Compute percentage stats per game
        fgm_col = 'field_goals_made' if 'field_goals_made' in d.columns else None
        fga_col = 'field_goals_attempted' if 'field_goals_attempted' in d.columns else None
        if fgm_col and fga_col:
            fgm = pd.to_numeric(d[fgm_col], errors='coerce')
            fga = pd.to_numeric(d[fga_col], errors='coerce')
            d['FG_PCT'] = (fgm / fga).where(fga > 0, 0.0)
        else:
            d['FG_PCT'] = 0.0
        fg3m = pd.to_numeric(d.get('three_point_field_goals_made', pd.Series(dtype=float)), errors='coerce')
        fg3a = pd.to_numeric(d.get('three_point_field_goals_attempted', pd.Series(dtype=float)), errors='coerce')
        d['FG3_PCT'] = (fg3m / fg3a).where(fg3a > 0, 0.0)
        ftm = pd.to_numeric(d.get('free_throws_made', pd.Series(dtype=float)), errors='coerce')
        fta = pd.to_numeric(d.get('free_throws_attempted', pd.Series(dtype=float)), errors='coerce')
        d['FT_PCT'] = (ftm / fta).where(fta > 0, 0.0)

        # Per-minute rates for count stats, raw averages for pct stats
        result = {}
        mins = d['_min']
        for w in WINDOWS:
            wd = d.head(w)
            wd_mins = mins.head(w)
            stats = {}
            for stat in PLAYER_COUNT_COLS:
                if stat in wd.columns:
                    vals = pd.to_numeric(wd[stat], errors='coerce').fillna(0)
                    per_min = vals / wd_mins
                    stats[stat] = per_min.mean() if len(per_min) > 0 else 0.0
                else:
                    stats[stat] = 0.0
            for stat in PLAYER_PCT_COLS:
                if stat in wd.columns:
                    vals = pd.to_numeric(wd[stat], errors='coerce').dropna()
                    stats[stat] = vals.mean() if len(vals) > 0 else 0.0
                else:
                    stats[stat] = 0.0
            result[w] = stats
        return result

    # Process new games
    new_games_t3 = training2_all[training2_all['game_id'].astype(str).isin(new_ids_t3)]
    output_rows = []
    dropped = 0

    for idx, row in new_games_t3.iterrows():
        game_id = str(row['game_id'])
        game_date = row['game_date']
        new_row = row.to_dict()

        # Find roster
        if game_id not in game_id_to_roster:
            dropped += 1
            continue

        try:
            roster_df = pd.read_csv(game_id_to_roster[game_id],
                                    dtype={'athlete_id': str, 'team_id': str})
        except Exception:
            dropped += 1
            continue

        if roster_df.empty:
            dropped += 1
            continue

        # Use ALL roster players (no current-game minutes filter).
        # This is pre-game information: we know who's on the roster.
        pid_col = 'athlete_id' if 'athlete_id' in roster_df.columns else None
        tid_col = 'team_id' if 'team_id' in roster_df.columns else None
        if not pid_col or not tid_col:
            dropped += 1
            continue

        # Match teams to home/away
        home_abbr = canonical(row['home_team'])
        away_abbr = canonical(row['away_team'])
        unique_tids = roster_df[tid_col].dropna().unique()
        home_tid = away_tid = None
        for t in unique_tids:
            abbr = team_id_to_abbr.get(str(t))
            if abbr == home_abbr:
                home_tid = str(t)
            elif abbr == away_abbr:
                away_tid = str(t)

        if not home_tid or not away_tid:
            dropped += 1
            continue

        # Initialize aggregated stats
        away_agg = {w: {s: 0.0 for s in PLAYER_STAT_COLS} for w in WINDOWS}
        home_agg = {w: {s: 0.0 for s in PLAYER_STAT_COLS} for w in WINDOWS}
        has_away = has_home = False

        for side, side_tid, agg, has_flag_name in [
            ('away', away_tid, away_agg, 'has_away'),
            ('home', home_tid, home_agg, 'has_home'),
        ]:
            side_players = roster_df[roster_df[tid_col].astype(str) == side_tid].copy()
            if side_players.empty:
                continue

            # Weight by PRIOR average minutes (no current-game data used)
            prior_mins = [get_prior_avg_minutes(str(p[pid_col]), game_date)
                          for _, p in side_players.iterrows()]
            # Only keep players with nonzero prior history
            side_players["_prior_min"] = prior_mins
            side_players = side_players[side_players["_prior_min"] > 0].copy()
            total_prior = side_players["_prior_min"].sum()
            if total_prior > 0:
                weights = (side_players["_prior_min"] / total_prior).tolist()
            else:
                continue

            for (_, player), weight in zip(side_players.iterrows(), weights):
                if weight == 0.0:
                    continue
                history = get_player_rolling(str(player[pid_col]), game_id, game_date)
                if history is None:
                    continue
                if side == 'away':
                    has_away = True
                else:
                    has_home = True
                for w in WINDOWS:
                    for stat in PLAYER_STAT_COLS:
                        agg[w][stat] += weight * history[w][stat]

        if not has_away or not has_home:
            dropped += 1
            continue

        # Add aggregated rolling stats to row
        for w in WINDOWS:
            for stat in PLAYER_STAT_COLS:
                new_row[f'away_{w}g_player_{stat}'] = away_agg[w][stat]
                new_row[f'home_{w}g_player_{stat}'] = home_agg[w][stat]

        output_rows.append(new_row)

        if (len(output_rows) + dropped) % 10 == 0:
            print(f"Processed {len(output_rows) + dropped}/{len(new_games_t3)} games, "
                  f"{len(output_rows)} kept, {dropped} dropped...")

    if not output_rows:
        print(f"No new games could be processed ({dropped} dropped)")
    else:
        new_df = pd.DataFrame(output_rows)
        new_df = new_df.reindex(columns=training3_existing.columns)
        new_df.to_csv(TRAINING3_CSV, mode='a', header=False, index=False)
        print(f"\nAppended {len(new_df)} game(s) to {TRAINING3_CSV}")
        print(f"  Dropped {dropped} game(s) (no roster or player data)")

        # Show sample to verify non-zero player stats
        sample_cols = ['game_id', 'home_team', 'away_team',
                       'away_5g_player_PTS', 'home_5g_player_PTS',
                       'away_10g_player_PTS', 'home_10g_player_PTS',
                       'away_20g_player_PTS', 'home_20g_player_PTS']
        available = [c for c in sample_cols if c in new_df.columns]
        print(f"\nSample rows:")
        print(new_df[available].head(5).to_string(index=False))

New games from today's batch: 16
Processed 10/16 games, 10 kept, 0 dropped...

Appended 16 game(s) to data/training_games3.csv
  Dropped 0 game(s) (no roster or player data)

Sample rows:
  game_id home_team away_team  away_5g_player_PTS  home_5g_player_PTS  away_10g_player_PTS  home_10g_player_PTS  away_20g_player_PTS  home_20g_player_PTS
401810989       CHI       PHX            0.502818            0.468220             0.484223             0.473342             0.479407             0.464048
401810988       BKN       WSH            0.502015            0.434240             0.473036             0.450714             0.460642             0.422352
401810987       BOS       TOR            0.507504            0.505887             0.500146             0.475783             0.451195             0.472980
401810991       CLE       IND            0.431821            0.456683             0.417304             0.465555             0.404690             0.478614
401810994       OKC      UTAH            0

In [11]:
# ── Append new games to training_games4.csv ──
# Mirrors the transforms in data_preprocess.ipynb (Cells 21-29):
#   training_games3 → filter season, derive features, drop metadata → training_games4

TRAINING4_CSV = 'data/training_games4.csv'

training4_existing = pd.read_csv(TRAINING4_CSV, dtype={'game_id': str})
training3_all = pd.read_csv(TRAINING3_CSV, dtype={'game_id': str})

# Use new_games IDs to select only the freshly added rows
new_ids_t4 = set(new_games['game_id'].astype(str))
new_t3 = training3_all[training3_all['game_id'].astype(str).isin(new_ids_t4)].copy()

# Also skip any already in training_games4
existing_ids_t4 = set(training4_existing['game_id'].astype(str))
new_t3 = new_t3[~new_t3['game_id'].astype(str).isin(existing_ids_t4)]

if new_t3.empty:
    print("No new games to add to training_games4.csv.")
else:
    print(f"Processing {len(new_t3)} new game(s) for training_games4.csv")

    # 1. Filter to seasons >= 2008-09
    new_t3 = new_t3[new_t3['season'] >= '2008-09'].copy()
    if new_t3.empty:
        print("No games remaining after season filter.")
    else:
        # 2. Drop columns not in training_games4
        drop_cols = ['season', 'home_team_full', 'away_team_full',
                     'away_PLUS_MINUS', 'home_PLUS_MINUS']
        new_t3.drop(columns=[c for c in drop_cols if c in new_t3.columns],
                    inplace=True, errors='ignore')

        # 3. Compute eFG from recency-weighted stats
        new_t3['away_eFG'] = (new_t3['away_FGM'] + 0.5 * new_t3['away_FG3M']) / new_t3['away_FGA'] * 100
        new_t3['home_eFG'] = (new_t3['home_FGM'] + 0.5 * new_t3['home_FG3M']) / new_t3['home_FGA'] * 100

        # 4. Win percentages
        new_t3['home_win_pct'] = new_t3['home_wins'] / (new_t3['home_wins'] + new_t3['home_losses'] + 0.01)
        new_t3['away_win_pct'] = new_t3['away_wins'] / (new_t3['away_wins'] + new_t3['away_losses'] + 0.01)

        # 5. Team ID encoding (alphabetical, 0-29)
        TEAM_TO_ID = {abbr: i for i, abbr in enumerate(sorted([
            'ATL', 'BOS', 'BKN', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW',
            'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK',
            'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS'
        ]))}
        new_t3['home_team_id'] = new_t3['home_team'].apply(canonical).map(TEAM_TO_ID)
        new_t3['away_team_id'] = new_t3['away_team'].apply(canonical).map(TEAM_TO_ID)

        # 6. Timezone difference
        CITY_TZ = {
            'Atlanta': -5, 'Boston': -5, 'Brooklyn': -5, 'Charlotte': -5,
            'Cleveland': -5, 'Detroit': -5, 'Indiana': -5, 'Miami': -5,
            'New Jersey': -5, 'New York': -5, 'Orlando': -5, 'Philadelphia': -5,
            'Toronto': -5, 'Washington': -5,
            'Chicago': -6, 'Dallas': -6, 'Houston': -6, 'Memphis': -6,
            'Milwaukee': -6, 'Minnesota': -6, 'New Orleans': -6,
            'NO/Oklahoma City': -6, 'Oklahoma City': -6, 'San Antonio': -6,
            'Denver': -7, 'Utah': -7, 'Phoenix': -7,
            'Golden State': -8, 'LA': -8, 'Los Angeles': -8,
            'Portland': -8, 'Sacramento': -8, 'Seattle': -8,
        }
        home_tz = new_t3['home_TEAM_CITY'].str.strip().map(CITY_TZ)
        away_tz = new_t3['away_TEAM_CITY'].str.strip().map(CITY_TZ)
        new_t3['tz_diff'] = home_tz - away_tz
        new_t3.drop(columns=['home_TEAM_CITY', 'away_TEAM_CITY'], inplace=True)

        # 7. Compute r10 eFG (rolling 10-game effective field goal %)
        new_t3['away_r10_eFG'] = (new_t3['away_r10_FGM'] + 0.5 * new_t3['away_r10_FG3M']) / new_t3['away_r10_FGA'].clip(lower=1) * 100
        new_t3['home_r10_eFG'] = (new_t3['home_r10_FGM'] + 0.5 * new_t3['home_r10_FG3M']) / new_t3['home_r10_FGA'].clip(lower=1) * 100

        # 8. Drop r10 PLUS_MINUS (fully NA)
        new_t3.drop(columns=[c for c in ['home_r10_PLUS_MINUS', 'away_r10_PLUS_MINUS'] if c in new_t3.columns],
                    inplace=True, errors='ignore')

        # 9. Drop rows where r10 fields are NA (early-season cold start)
        r10_cols = [c for c in new_t3.columns if 'r10' in c]
        before = len(new_t3)
        new_t3.dropna(subset=r10_cols, inplace=True)
        if before - len(new_t3) > 0:
            print(f"Dropped {before - len(new_t3)} row(s) with NA r10 fields")

        # 10. Score diff, drop actual PTS
        new_t3['score_diff'] = new_t3['home_PTS_actual'] - new_t3['away_PTS_actual']
        new_t3.drop(columns=['home_PTS_actual', 'away_PTS_actual'], inplace=True)

        # Align columns to existing training_games4.csv and append
        new_t3 = new_t3.reindex(columns=training4_existing.columns)
        new_t3.to_csv(TRAINING4_CSV, mode='a', header=False, index=False)
        print(f"Appended {len(new_t3)} game(s) to {TRAINING4_CSV}")
        print(new_t3[['game_id', 'game_date', 'home_team', 'away_team',
                       'home_eFG', 'away_eFG', 'home_win_pct', 'away_win_pct',
                       'tz_diff', 'score_diff']].to_string(index=False))

Processing 16 new game(s) for training_games4.csv
Appended 16 game(s) to data/training_games4.csv
  game_id  game_date home_team away_team  home_eFG  away_eFG  home_win_pct  away_win_pct  tz_diff  score_diff
401810989 2026-04-05       CHI       PHX 54.150083 53.999827      0.376574      0.545384        1         -10
401810988 2026-04-05       BKN       WSH 51.454527 54.779240      0.233736      0.220751        0           6
401810987 2026-04-05       BOS       TOR 56.065793 56.985048      0.675237      0.558369        0          14
401810991 2026-04-05       CLE       IND 57.781253 58.656460      0.623296      0.233736        0           9
401810994 2026-04-05       OKC      UTAH 55.849019 52.994346      0.792105      0.269196        1          35
401810995 2026-04-05       DAL       LAL 51.166602 57.556606      0.311648      0.649266        2           6
401810992 2026-04-05       MIN       CHA 52.495380 56.956096      0.597325      0.538393       -1         -14
401810996 2026-04-05  

In [12]:
# ── Fetch missing Kalshi trade data for new games ──
# Imports fetch_kalshi module and runs the same logic as the CLI script,
# but only for games that have PBP files and don't already have Kalshi data.

from dotenv import load_dotenv
from pathlib import Path as _Path

# Load .env from project root (one level up from nba/)
_project_root = _Path('.').resolve().parent
load_dotenv(_project_root / ".env")

sys.path.insert(0, str(_Path('.').resolve() / 'scripts'))
from fetch_kalshi import (
    KalshiClient, process_one_game, get_output_path,
    parse_pbp_filename, get_game_date_from_pbp,
    ALL_STAR_CODES,
)

KALSHI_PBP_DIR = Path("data/games_live/2026")
KALSHI_OUT_DIR = Path("data/kalshi_live")
KALSHI_DELAY = 0.5  # seconds between games

# Initialize Kalshi client
try:
    kalshi_client = KalshiClient()
    print("Kalshi client initialized")
except Exception as e:
    kalshi_client = None
    print(f"WARNING: Could not initialize Kalshi client: {e}")
    print("Skipping Kalshi data fetch (set KALSHI_API_KEY in .env)")

if kalshi_client is not None:
    # Find all PBP files
    pbp_files = sorted(KALSHI_PBP_DIR.glob("*.csv"))
    print(f"PBP files in {KALSHI_PBP_DIR}: {len(pbp_files)}")

    processed = 0
    skipped = 0
    errors = 0
    skipped_allstar = 0

    for pbp_file in pbp_files:
        pbp_path = str(pbp_file)

        # Check if output already exists
        out_path = get_output_path(pbp_path, str(KALSHI_OUT_DIR))
        if os.path.exists(out_path):
            skipped += 1
            continue

        # Parse game info
        try:
            espn_id, away_code, home_code = parse_pbp_filename(pbp_path)
        except ValueError:
            continue

        # Skip All-Star games
        if away_code in ALL_STAR_CODES or home_code in ALL_STAR_CODES:
            skipped_allstar += 1
            continue

        # Get game date
        try:
            game_date = get_game_date_from_pbp(pbp_path)
        except Exception:
            errors += 1
            continue

        print(f"  [{processed + 1}] {game_date.date()} {away_code} @ {home_code} ...", end=" ")

        try:
            out = process_one_game(
                pbp_path, kalshi_client, game_date,
                away_code, home_code, out_dir=str(KALSHI_OUT_DIR)
            )
            print(f"OK")
            processed += 1
        except Exception as e:
            print(f"ERROR: {e}")
            errors += 1

        time.sleep(KALSHI_DELAY)

    print(f"\nKalshi fetch summary:")
    print(f"  New: {processed}, Skipped (exist): {skipped}, "
          f"Skipped (All-Star): {skipped_allstar}, Errors: {errors}")

Kalshi client initialized
PBP files in data/games_live/2026: 1179
  [1] 2026-04-05 TOR @ BOS ...   Game window: 2026-04-05 19:12:03+00:00 to 2026-04-05 21:50:59+00:00
  Home ticker: KXNBAGAME-26APR05TORBOS-BOS
  Away ticker: KXNBAGAME-26APR05TORBOS-TOR
  Checking if markets exist...
    Home market found: Toronto at Boston Winner?
      Status: finalized, Volume: None, Open Interest: None
    Away market found: Toronto at Boston Winner?
      Status: finalized, Volume: None, Open Interest: None
  Fetching home team trades...
    [DEBUG] Fetching trades for KXNBAGAME-26APR05TORBOS-BOS
    [DEBUG] Time range: 2026-04-05 19:12:03+00:00 to 2026-04-05 21:50:59+00:00
    [DEBUG] Unix timestamps: 1775416323 to 1775425859
    [DEBUG] GET https://api.elections.kalshi.com/trade-api/v2/markets/trades
    [DEBUG] Params: {'limit': 1000, 'min_ts': 1775416323, 'max_ts': 1775425859, 'ticker': 'KXNBAGAME-26APR05TORBOS-BOS'}
    [DEBUG] Status: 200
    [DEBUG] Response: {"cursor":"EhIKEAJW8l20UEl3IEEXS

In [16]:
# ── Add Kalshi pregame implied probabilities to games_predictions CSVs ──
# For each game missing a kalshi_pregame_wp value, compute the median implied
# home-win probability from Kalshi trades in the 5 minutes before tipoff.
# Tipoff is determined from the first PBP wallclock timestamp.

from pathlib import Path

KALSHI_DIR = Path("data/kalshi_live")
PBP_DIR = Path("data/games_live/2026")
PREDICTIONS_FILES = [
    Path("data/games_predictions.csv"),
    Path("data/games_predictions_deploy.csv"),
]

# delete the existing kalshi_pregame_wp column in the predictions files

df = pd.read_csv(PREDICTIONS_FILES[1])

if "kalshi_pregame_wp" in df.columns:
    df = df.drop(columns=["kalshi_pregame_wp"])
    df.to_csv(PREDICTIONS_FILES[1], index=False)

def _get_tipoff_utc(game_id: str) -> pd.Timestamp:
    """Get tipoff time from first PBP wallclock for a game."""
    hits = list(PBP_DIR.glob(f"{game_id}_*.csv"))
    if not hits:
        return None
    pbp = pd.read_csv(hits[0], nrows=1, usecols=["wallclock"])
    return pd.to_datetime(pbp["wallclock"].iloc[0], utc=True)


def _get_kalshi_pregame_wp(game_id: str, tipoff_utc: pd.Timestamp) -> float:
    """Compute median home-win implied probability from Kalshi trades
    in the 5-minute window before tipoff."""
    hits = list(KALSHI_DIR.glob(f"{game_id}_*_kalshi_*.csv"))
    if not hits:
        return np.nan

    k = pd.read_csv(hits[0])
    k["timestamp"] = pd.to_datetime(k["timestamp"], format="ISO8601", utc=True)

    # 5-minute window before tipoff
    window_start = tipoff_utc - pd.Timedelta(minutes=5)
    pre = k[(k["timestamp"] >= window_start) & (k["timestamp"] < tipoff_utc)]

    if len(pre) < 2:
        return np.nan

    if pre.empty:
        return np.nan

    # Compute implied home-win probability from bid/ask midpoints
    home_mid = (pre["home_high_cents"] + pre["home_low_cents"]) / 2
    away_mid = (pre["away_high_cents"] + pre["away_low_cents"]) / 2

    # Kalshi data format changed mid-season: earlier files use 0-1 probabilities,
    # later files use actual cents (0-100). Normalize to 0-1.
    if home_mid.dropna().max() > 1 or away_mid.dropna().max() > 1:
        home_mid = home_mid / 100
        away_mid = away_mid / 100

    implied = np.where(
        pre["home_volume"] > 0, home_mid,
        np.where(pre["away_volume"] > 0, 1 - away_mid, np.nan)
    )

    valid = pd.Series(implied).dropna()
    if valid.empty:
        return np.nan

    return float(valid.median())


for pred_path in PREDICTIONS_FILES:
    if not pred_path.exists():
        print(f"SKIP: {pred_path} not found")
        continue

    df = pd.read_csv(pred_path, dtype={"game_id": str})
    df["game_id"] = df["game_id"].str.strip()

    # Add column if it doesn't exist
    if "kalshi_pregame_wp" not in df.columns:
        df["kalshi_pregame_wp"] = np.nan

    # Find rows missing Kalshi data
    missing_mask = df["kalshi_pregame_wp"].isna()
    missing_ids = df.loc[missing_mask, "game_id"].unique()
    print(f"\n{pred_path.name}: {len(df)} games, {missing_mask.sum()} missing Kalshi pregame")

    if len(missing_ids) == 0:
        print("  Nothing to fill!")
        continue

    filled = 0
    no_data = 0

    for i, gid in enumerate(missing_ids):
        if (i + 1) % 100 == 0:
            print(f"  Processing {i+1}/{len(missing_ids)} ...")

        tipoff = _get_tipoff_utc(gid)
        if tipoff is None:
            no_data += 1
            continue

        wp = _get_kalshi_pregame_wp(gid, tipoff)
        if np.isnan(wp):
            no_data += 1
            continue

        df.loc[df["game_id"] == gid, "kalshi_pregame_wp"] = wp
        filled += 1

    # Clip only non-NaN values to valid probability range
    mask = df["kalshi_pregame_wp"].notna()
    df.loc[mask, "kalshi_pregame_wp"] = df.loc[mask, "kalshi_pregame_wp"].clip(0.001, 0.999)

    df.to_csv(pred_path, index=False)
    total_filled = df["kalshi_pregame_wp"].notna().sum()
    print(f"  Filled {filled} new games ({no_data} with no Kalshi/PBP data)")
    print(f"  Total with Kalshi pregame: {total_filled}/{len(df)}")
    print(f"  Saved to {pred_path}")

SKIP: data/games_predictions.csv not found

games_predictions_deploy.csv: 17258 games, 17258 missing Kalshi pregame
  Processing 100/17258 ...
  Processing 200/17258 ...
  Processing 300/17258 ...
  Processing 400/17258 ...
  Processing 500/17258 ...
  Processing 600/17258 ...
  Processing 700/17258 ...
  Processing 800/17258 ...
  Processing 900/17258 ...
  Processing 1000/17258 ...
  Processing 1100/17258 ...
  Processing 1200/17258 ...
  Processing 1300/17258 ...
  Processing 1400/17258 ...
  Processing 1500/17258 ...
  Processing 1600/17258 ...
  Processing 1700/17258 ...
  Processing 1800/17258 ...
  Processing 1900/17258 ...
  Processing 2000/17258 ...
  Processing 2100/17258 ...
  Processing 2200/17258 ...
  Processing 2300/17258 ...
  Processing 2400/17258 ...
  Processing 2500/17258 ...
  Processing 2600/17258 ...
  Processing 2700/17258 ...
  Processing 2800/17258 ...
  Processing 2900/17258 ...
  Processing 3000/17258 ...
  Processing 3100/17258 ...
  Processing 3200/17258 .

In [18]:
# ── Verification: check for missing data in the current season ──

VERIFY_SEASON_START = '2025-10-01'
VERIFY_SEASON_LABEL = '2025-26'

print(f'=== {VERIFY_SEASON_LABEL} Season Data Verification ===\n')

# --- games_predictions.csv ---
for pred_file in ['data/games_predictions.csv', 'data/games_predictions_deploy.csv']:
    pred = pd.read_csv(pred_file, dtype={'game_id': str})
    pred['game_date'] = pd.to_datetime(pred['game_date'])
    season = pred[pred['game_date'] >= VERIFY_SEASON_START]

    print(f'{Path(pred_file).name}:')
    print(f'  Total games: {len(season)}')
    print(f'  Date range: {season.game_date.min().date()} to {season.game_date.max().date()}')

    if 'kalshi_pregame_wp' in season.columns:
        has = season['kalshi_pregame_wp'].notna().sum()
        missing_k = season[season['kalshi_pregame_wp'].isna()]
        print(f'  Kalshi prior: {has}/{len(season)} ({has/len(season)*100:.1f}%)')
        if len(missing_k) > 0:
            print(f'  Missing Kalshi ({len(missing_k)}):')
            for _, r in missing_k.iterrows():
                print(f'    {r.game_id}  {r.game_date.date()}  {r.away_team} @ {r.home_team}')
    else:
        print(f'  Kalshi prior: column not yet added')

    na_prior = season['prior_home_wp'].isna().sum()
    na_actual = season['home_win_actual'].isna().sum()
    na_score = season['score_diff'].isna().sum()
    if na_prior + na_actual + na_score > 0:
        print(f'  WARNING: NA values — prior_home_wp: {na_prior}, home_win_actual: {na_actual}, score_diff: {na_score}')
    else:
        print(f'  Core columns: all complete')
    print()

# --- training_games4.csv ---
t4 = pd.read_csv('data/training_games4.csv', dtype={'game_id': str})
t4['game_date'] = pd.to_datetime(t4['game_date'])
t4_season = t4[t4['game_date'] >= VERIFY_SEASON_START]
na_cols = t4_season.isna().sum()
na_cols = na_cols[na_cols > 0]
print(f'training_games4.csv:')
print(f'  {VERIFY_SEASON_LABEL} games: {len(t4_season)}')
if len(na_cols) > 0:
    print(f'  Columns with NAs:')
    for col, cnt in na_cols.items():
        print(f'    {col}: {cnt}')
else:
    print(f'  All columns complete')
print()

# --- PBP and Kalshi file coverage ---
import os
pbp_ids = set(f.split('_')[0] for f in os.listdir('data/games_live/2026/') if f.endswith('.csv'))
kalshi_ids = set(f.split('_')[0] for f in os.listdir('data/kalshi_live/') if f.endswith('.csv'))
pred_ids = set(season['game_id'].str.strip())

print(f'File coverage:')
print(f'  PBP files (games_live/2026/): {len(pbp_ids)}')
print(f'  Kalshi files (kalshi_live/): {len(kalshi_ids)}')
print(f'  Predictions matching PBP: {len(pred_ids & pbp_ids)}/{len(pred_ids)}')
print(f'  Predictions matching Kalshi: {len(pred_ids & kalshi_ids)}/{len(pred_ids)}')

=== 2025-26 Season Data Verification ===



FileNotFoundError: [Errno 2] No such file or directory: 'data/games_predictions.csv'